# QSV2G — Stage 02: Security Analysis and Attack Evaluation

**Research question:** Which Section III security/privacy requirements are demonstrated by the software prototype under explicit assumptions, which simulated attacks are rejected, and which limitations remain?

**Inputs:** Original Section IV reference Python implementation, without functional changes, SHA-256 `2c86ca9eecbc5925100722119f061e640df03f993e14909b3078b23406330758`; Section IV LaTeX reference SHA-256 `4ca43b1f6debd2e2bf314ff9adf74f0ad08ba1702e1055eca1943b9533290ef2`.

**Important distinctions:**
- An expected `REJECTED` outcome is evidence of a tested rejection only; it is **not a security proof**.
- An expected `EXPOSED` finding is a successfully *demonstrated limitation*, **not a successful defense**.
- `NOT_ESTABLISHED` means the simulator does not measure the claimed security property.
- Any `ERROR` or mismatch triggers `REVIEW_REQUIRED` in the summary and is retained in the exported artifacts.
- Quantum operations use ideal 27-dimensional statevectors. Keys, inter-authority channels, quantum memories, policy oracles, crash-persistent storage, and hardware power controls have assumptions not verified here.

**Execution:** Kaggle CPU is sufficient; accelerator `None`; internet enabled *only if packages need installation*. The output ZIP is `qsv2g_stage02_security_results.zip` in Kaggle Outputs. No GitHub changes are made by this notebook.

## 0. Kaggle dependencies, environment, and real PQC provider

**Fix for Kaggle `ModuleNotFoundError: No module named 'pqcrypto'`:** The next setup cell checks both supported backends. It installs native `pqcrypto` automatically **if neither its ML-KEM-768/ML-DSA-65 modules nor a compatible OpenSSL 3.5+ backend is available**. There is no fake or classical cryptographic substitute.

In Kaggle, choose **Notebook settings → Internet → On** if package installation is needed. Select **Accelerator: None (CPU)**; GPU is not required for Stage 02. Run the notebook **from the first cell**. If Kaggle restricts Internet, install `pqcrypto` in an Internet-enabled notebook environment and restart the session, or use an environment with OpenSSL 3.5+ PQC support.

The exact source-hash check in Section 1 is preserved; this setup only ensures its cryptographic dependencies are available.

In [ ]:
import sys, os, pathlib, subprocess, importlib, importlib.util, platform, json, hashlib, time, collections
import shutil

# Install packages only when missing, using the running notebook kernel's Python.
def _install_pip_package(package):
    cmd = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-input', '-q', package]
    print('Installing required package:', package, flush=True)
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=240)
    if result.returncode != 0:
        err = (result.stderr or result.stdout or 'Unknown package-installation error').strip()
        raise RuntimeError(
            f'Could not install {package}. In Kaggle enable Internet in Notebook settings, '
            f'then restart the session and Run All. pip error: {err[-1600:]}')
    importlib.invalidate_caches()

for pkg, mod in [('numpy','numpy'), ('pandas','pandas'), ('cryptography','cryptography')]:
    if importlib.util.find_spec(mod) is None:
        _install_pip_package(pkg)

# Prefer the standardized native pqcrypto provider when already present.
def _native_pqc_present():
    try:
        from pqcrypto.kem import ml_kem_768
        from pqcrypto.sign import ml_dsa_65
        return True
    except (ImportError, ModuleNotFoundError):
        return False

# The unmodified reference implementation can also use OpenSSL 3.5+.
def _openssl_pqc_present():
    exe = shutil.which('openssl')
    if not exe:
        return False
    try:
        kem = subprocess.run([exe, 'list', '-kem-algorithms'], capture_output=True, text=True, timeout=20)
        sig = subprocess.run([exe, 'list', '-signature-algorithms'], capture_output=True, text=True, timeout=20)
        return 'ML-KEM-768' in kem.stdout and 'ML-DSA-65' in sig.stdout
    except (OSError, subprocess.TimeoutExpired):
        return False

if not _native_pqc_present() and not _openssl_pqc_present():
    _install_pip_package('pqcrypto')

# Re-check after installation. Fail early instead of silently changing cryptography.
if not (_native_pqc_present() or _openssl_pqc_present()):
    raise RuntimeError(
        'Neither pqcrypto (ML-KEM-768 and ML-DSA-65) nor OpenSSL 3.5+ PQC is available. '
        'In Kaggle turn Internet ON, restart the notebook session, and Run All.')

import numpy as np, pandas as pd
import itertools, secrets, random, traceback
from unittest.mock import patch
from importlib import metadata
ON_KAGGLE = pathlib.Path('/kaggle/working').is_dir()
WORK = pathlib.Path('/kaggle/working') if ON_KAGGLE else pathlib.Path.cwd()
OUT = WORK / 'qsv2g_stage02_security_results'
# Reproducible reruns: clear only this notebook's known output directory.
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)
print('Environment:', platform.platform(), 'Python:', platform.python_version())
print('PQC availability: pqcrypto =', _native_pqc_present(), '; OpenSSL 3.5+ =', _openssl_pqc_present())
print('Outputs:', OUT)


## 1. Exact unmodified QSV2G reference implementation
The next cell executes the byte-for-byte source taken from the previous full-protocol and final-validation notebooks. A SHA-256 check aborts if it is altered. **Tests below may deliberately mutate copies of protocol object state to model compromised principals**; that is explicitly distinguished from unauthenticated network tampering.

In [ ]:
REFERENCE_SHA256 = "2c86ca9eecbc5925100722119f061e640df03f993e14909b3078b23406330758"
REFERENCE_CORE = '"""QSV2G Section IV reference prototype (research simulation, NOT deployed secure system).\nSource: QSV2G_main8_SectionIV_Revised.tex.\nQuantum: exact 27D three-qutrit statevectors with sampled pair measurements.\nPQC: actual ML-KEM-768 / ML-DSA-65 via pqcrypto OR OpenSSL >= 3.5.\nThis is not malicious-secure verifiable QSS. Authenticated channels, secure enrollment,\nquantum memory and two-authority honesty are simulation assumptions.\n"""\nfrom __future__ import annotations\nimport base64\nimport hashlib\nimport hmac\nimport json\nimport math\nimport os\nimport secrets\nimport shutil\nimport subprocess\nimport tempfile\nimport time\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any\nimport numpy as np\nfrom cryptography.hazmat.primitives.ciphers.aead import AESGCM\n\nLABEL = \'QSV2G-REFERENCE-2026\'\nAUTHORITIES = (\'A\',\'B\',\'C\')\nSLOT = {\'A\':0,\'B\':1,\'C\':2}\n\nclass ProtocolAbort(Exception):\n    pass\n\ndef require(condition, explanation):\n    if not condition:\n        raise ProtocolAbort(explanation)\n\ndef json_obj(o):\n    if isinstance(o, bytes): return {\'__type__\':\'bytes\', \'base64\':base64.b64encode(o).decode(\'ascii\')}\n    if isinstance(o, tuple): return {\'__type__\':\'tuple\', \'items\':[json_obj(x) for x in o]}\n    if isinstance(o, list): return [json_obj(x) for x in o]\n    if isinstance(o, dict): return {str(k):json_obj(v) for k,v in sorted(o.items())}\n    if isinstance(o, (str,int,bool)) or o is None: return o\n    raise TypeError(f\'Unsupported field {type(o).__name__}: floating-point values are intentionally disallowed\')\n\ndef canon(x):\n    return json.dumps(json_obj(x), sort_keys=True, separators=(\',\',\':\'), ensure_ascii=False).encode(\'utf-8\')\n\ndef parse_canon(raw):\n    def restore(x):\n        if isinstance(x, list): return [restore(y) for y in x]\n        if isinstance(x, dict):\n            if x.get(\'__type__\')==\'bytes\' and set(x)=={\'__type__\',\'base64\'}:\n                return base64.b64decode(x[\'base64\'],validate=True)\n            if x.get(\'__type__\')==\'tuple\' and set(x)=={\'__type__\',\'items\'}:\n                return tuple(restore(y) for y in x[\'items\'])\n            return {k:restore(v) for k,v in x.items()}\n        return x\n    return restore(json.loads(raw.decode(\'utf-8\')))\n\ndef encode(*fields):\n    """Fixed ordering, deterministic, typed-length-separated encoding."""\n    out=bytearray()\n    for f in fields:\n        chunk=canon(f)\n        out += len(chunk).to_bytes(4,\'big\')+chunk\n    return bytes(out)\n\ndef H(*fields): return hashlib.sha256(encode(*fields)).digest()\ndef mac(k, *fields): return hmac.new(k, encode(*fields), \'sha256\').digest()\ndef ct_eq(a,b): return hmac.compare_digest(a,b)\ndef hkdf_extract(salt, ikm): return hmac.new(salt, ikm, \'sha256\').digest()\ndef hkdf_expand(prk, info, n=32):\n    out=b\'\'; t=b\'\'\n    for i in range(1,math.ceil(n/32)+1):\n        t=hmac.new(prk, t+info+bytes([i]), \'sha256\').digest(); out+=t\n    return out[:n]\n\ndef digest_for(label, *items): return H(label, *items)\n\n# ---- Quantum statevectors ----\nI2=np.eye(2,dtype=np.complex128)\nX2=np.array([[0,1],[1,0]],complex)\nZ2=np.diag([1,-1]).astype(complex)\nH2=np.array([[1,1],[1,-1]],complex)/np.sqrt(2)\nCNOT=np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]],complex)\nBELL=np.array([1,0,0,1],complex)/np.sqrt(2)\n\ndef dense_encode_decode(two_bits):\n    a,b=two_bits\n    state=np.kron(np.linalg.matrix_power(Z2,a)@np.linalg.matrix_power(X2,b),I2)@BELL\n    final=np.kron(H2,I2)@CNOT@state\n    probabilities=np.abs(final)**2\n    idx=int(np.argmax(probabilities))\n    require(np.isclose(probabilities[idx],1,atol=1e-12),\'Bell decoding failure\')\n    return (idx//2,idx%2)\n\ndef dense_transport_tag(tag):\n    recovered=bytearray()\n    bits=\'\'.join(f\'{byte:08b}\' for byte in tag)\n    out=\'\'\n    for k in range(0,len(bits),2):\n        aa,bb=dense_encode_decode((int(bits[k]),int(bits[k+1])))\n        out+=str(aa)+str(bb)\n    recovered.extend(int(out[j:j+8],2) for j in range(0,len(out),8))\n    return bytes(recovered)\n\nOMEGA=np.exp(2j*np.pi/3)\nX3=np.roll(np.eye(3,dtype=complex),1,axis=0)\nZ3=np.diag([OMEGA**j for j in range(3)])\n\n\ndef qss_state(s):\n    """|psi(s)> = sum_r |r,r+s,r+2s> /sqrt(3)."""\n    require(s in (0,1,2),\'Invalid trit\')\n    state=np.zeros((3,3,3),dtype=complex)\n    for r in range(3): state[r,(r+s)%3,(r+2*s)%3]=1/np.sqrt(3)\n    return state\n\ndef reduced_density(psi, kept):\n    moved=np.moveaxis(psi,kept,0).reshape(3,-1)\n    return moved@moved.conj().T\n\ndef measure_pair(psi, pair, rng, x_fault=False):\n    """Sample joint computational measurement of two shares from a 3-qutrit statevector.\n    x_fault applies a qutrit-X error to the SECOND selected share before measuring.\n    """\n    u,v=[SLOT[p] for p in pair]\n    state=np.moveaxis(psi,(u,v),(0,1))\n    probs=(np.abs(state)**2).sum(axis=2).reshape(-1)\n    idx=int(rng.choice(9,p=probs/probs.sum()))\n    xu,xv=divmod(idx,3)\n    if x_fault: xv=(xv+1)%3\n    return xu,xv\n\ndef recover_symbol(pair,xu,xv):\n    denom=(SLOT[pair[1]]-SLOT[pair[0]])%3\n    return ((xv-xu)*pow(denom,-1,3))%3\n\ndef qss_consistency():\n    Xa=np.kron(np.kron(X3,X3),X3)\n    Za=np.kron(np.kron(Z3,Z3),Z3)\n    vals=[]\n    for s in range(3):\n        psi=qss_state(s)\n        psi_flat=psi.reshape(-1)\n        vals.append({\n            \'s\':s,\n            \'norm\':float(np.vdot(psi_flat,psi_flat).real),\n            \'x_stabilizer_error\':float(np.linalg.norm(Xa@psi_flat-psi_flat)),\n            \'z_stabilizer_error\':float(np.linalg.norm(Za@psi_flat-psi_flat)),\n            \'one_share_distances\':[float(np.linalg.norm(reduced_density(psi,b)-np.eye(3)/3)) for b in range(3)],\n        })\n    return vals\n\n# ---- Actual standardized post-quantum cryptography, no silent fallbacks ----\nclass PQCrypto:\n    def __init__(self):\n        self.kind=None\n        try:\n            from pqcrypto.kem import ml_kem_768\n            from pqcrypto.sign import ml_dsa_65\n            self.kem=ml_kem_768; self.sig=ml_dsa_65\n            self.kind=\'pqcrypto native ML-KEM-768 + ML-DSA-65\'\n        except (ImportError,ModuleNotFoundError):\n            exe=shutil.which(\'openssl\')\n            if exe:\n                test=subprocess.run([exe,\'list\',\'-kem-algorithms\'],capture_output=True,text=True)\n                sig=subprocess.run([exe,\'list\',\'-signature-algorithms\'],capture_output=True,text=True)\n                if \'ML-KEM-768\' in test.stdout and \'ML-DSA-65\' in sig.stdout:\n                    self.kind=\'OpenSSL 3.5+ ML-KEM-768 + ML-DSA-65 (subprocess)\'\n                    self.openssl=exe\n            if self.kind is None:\n                raise RuntimeError(\'Real ML-KEM-768/ML-DSA-65 unavailable. Enable Kaggle Internet and pip install pqcrypto. No classical substitute will be used.\')\n    def _openssl_call(self, arguments, input_files=None, output_files=()):\n        with tempfile.TemporaryDirectory(prefix=\'qsv2g_pqc_\') as folder:\n            root=Path(folder)\n            for name,data in (input_files or {}).items(): (root/name).write_bytes(data)\n            argv=[self.openssl]+[str(root/(v[1:])) if isinstance(v,str) and v.startswith(\'@\') else str(v) for v in arguments]\n            p=subprocess.run(argv, stdout=subprocess.PIPE, stderr=subprocess.PIPE, cwd=root, timeout=30)\n            if p.returncode:\n                raise RuntimeError(\'OpenSSL cryptographic call failed: \'+p.stderr.decode(\'utf-8\',errors=\'replace\')[:500])\n            return (p.stdout, *[(root/x).read_bytes() for x in output_files])\n    def kem_keygen(self):\n        if self.kind.startswith(\'pqcrypto\'):\n            pub,priv=(self.kem.keygen() if hasattr(self.kem,\'keygen\') else self.kem.generate_keypair()); return pub,priv\n        # key generation and public-key extraction share one temporary directory\n        with tempfile.TemporaryDirectory(prefix=\'qsv2g_kem_\') as d:\n            sk=Path(d)/\'sk.pem\'; pk=Path(d)/\'pk.pem\'\n            subprocess.run([self.openssl,\'genpkey\',\'-algorithm\',\'ML-KEM-768\',\'-out\',str(sk)],check=True,capture_output=True)\n            subprocess.run([self.openssl,\'pkey\',\'-in\',str(sk),\'-pubout\',\'-out\',str(pk)],check=True,capture_output=True)\n            return pk.read_bytes(), sk.read_bytes()\n    def encaps(self,pub):\n        if self.kind.startswith(\'pqcrypto\'):\n            return self.kem.encaps(pub) if hasattr(self.kem,\'encaps\') else self.kem.encrypt(pub)\n        _,ct,ss=self._openssl_call([\'pkeyutl\',\'-encap\',\'-pubin\',\'-inkey\',\'@pk\',\'-out\',\'@ct\',\'-secret\',\'@ss\'], {\'pk\':pub},(\'ct\',\'ss\'))\n        return ct,ss\n    def decaps(self,priv,ct):\n        if self.kind.startswith(\'pqcrypto\'):\n            return self.kem.decaps(priv,ct) if hasattr(self.kem,\'decaps\') else self.kem.decrypt(priv,ct)\n        _,ss=self._openssl_call([\'pkeyutl\',\'-decap\',\'-inkey\',\'@sk\',\'-in\',\'@ct\',\'-secret\',\'@ss\'],{\'sk\':priv,\'ct\':ct},(\'ss\',))\n        return ss\n    def sig_keygen(self):\n        if self.kind.startswith(\'pqcrypto\'):\n            return self.sig.keygen() if hasattr(self.sig,\'keygen\') else self.sig.generate_keypair()\n        with tempfile.TemporaryDirectory(prefix=\'qsv2g_sig_\') as d:\n            sk=Path(d)/\'sk.pem\'; pk=Path(d)/\'pk.pem\'\n            subprocess.run([self.openssl,\'genpkey\',\'-algorithm\',\'ML-DSA-65\',\'-out\',str(sk)],check=True,capture_output=True)\n            subprocess.run([self.openssl,\'pkey\',\'-in\',str(sk),\'-pubout\',\'-out\',str(pk)],check=True,capture_output=True)\n            return pk.read_bytes(),sk.read_bytes()\n    def sign(self,priv,msg):\n        if self.kind.startswith(\'pqcrypto\'):\n            return self.sig.sign(priv,msg)\n        _,sig=self._openssl_call([\'pkeyutl\',\'-sign\',\'-inkey\',\'@sk\',\'-in\',\'@msg\',\'-out\',\'@sig\'],{\'sk\':priv,\'msg\':msg},(\'sig\',))\n        return sig\n    def verify(self,pub,msg,sig):\n        if self.kind.startswith(\'pqcrypto\'):\n            try:\n                result=self.sig.verify(pub,msg,sig)\n                return result is None or result is True\n            except Exception: return False  # InvalidSignatureError from pqcrypto 1.x, ValueError on older versions\n        try:\n            self._openssl_call([\'pkeyutl\',\'-verify\',\'-pubin\',\'-inkey\',\'@pk\',\'-in\',\'@msg\',\'-sigfile\',\'@sig\'],{\'pk\':pub,\'msg\':msg,\'sig\':sig})\n            return True\n        except RuntimeError: return False\n\n@dataclass\nclass Traffic:\n    key:bytes\n    send_counter:int=0\n    seen:set=field(default_factory=set)\n    def encrypt(self,pt:bytes,ad:bytes):\n        require(self.send_counter<2**96-1,\'AES-GCM counter exhaustion\')\n        nonce=self.send_counter.to_bytes(12,\'big\'); self.send_counter+=1\n        ct=AESGCM(self.key).encrypt(nonce,pt,ad)\n        return {\'nonce\':nonce,\'ciphertext\':ct}\n    def decrypt(self,pkt,ad):\n        nonce=pkt[\'nonce\']\n        require(len(nonce)==12 and nonce not in self.seen,\'duplicate or malformed AEAD nonce\')\n        try: pt=AESGCM(self.key).decrypt(nonce,pkt[\'ciphertext\'],ad)\n        except Exception as e: raise ProtocolAbort(\'AEAD authentication failure\') from e\n        self.seen.add(nonce)\n        return pt\n\n@dataclass\nclass QSSResource:\n    nu:str\n    dealer_secrets:dict\n    states:dict\n    commitments:dict\n    state:str=\'UNUSED\'\n    lock:tuple|None=None\n    def reserve(self,ac,pair):\n        require(self.state==\'UNUSED\',\'QSS resource already reserved or consumed\')\n        require(len(set(pair))==2 and all(a in AUTHORITIES for a in pair),\'pair invalid\')\n        self.state=\'RESERVED\'; self.lock=(ac,tuple(pair))\n    def reconstruct(self,ac,pair,rng,tamper=False):\n        require(self.state==\'RESERVED\' and self.lock==(ac,tuple(pair)),\'QSS reservation mismatch\')\n        recovered={a:[] for a in AUTHORITIES}\n        try:\n            for dealer in AUTHORITIES:\n                for k,psi in enumerate(self.states[dealer]):\n                    xu,xv=measure_pair(psi,pair,rng,x_fault=(tamper and dealer==\'A\' and k==0))\n                    recovered[dealer].append(recover_symbol(pair,xu,xv))\n            for dealer in AUTHORITIES:\n                expected=H(\'QSV2G-QSS-COM\',dealer,self.nu,recovered[dealer])\n                require(ct_eq(expected,self.commitments[dealer]),\'QSS reconstructed commitment mismatch\')\n            combined=[sum(recovered[a][k] for a in AUTHORITIES)%3 for k in range(len(recovered[\'A\']))]\n            return combined\n        finally:\n            self.state=\'CONSUMED\'\n            self.states.clear()  # simulated irreversible measurement / erasure\n\ndef new_qss_resource(L=162,rng=None):\n    rng=rng or np.random.default_rng(2026)\n    nu=secrets.token_hex(12)\n    dealer_secrets={a:[secrets.randbelow(3) for _ in range(L)] for a in AUTHORITIES}\n    states={a:[qss_state(s) for s in dealer_secrets[a]] for a in AUTHORITIES}\n    commits={a:H(\'QSV2G-QSS-COM\',a,nu,dealer_secrets[a]) for a in AUTHORITIES}\n    return QSSResource(nu,dealer_secrets,states,commits)\n\nclass QSV2G:\n    """Single-session strict reference implementation with explicit trusted authority simulation."""\n    def __init__(self,crypto=None, seed=2026,L=162,available=None):\n        self.pqc=crypto or PQCrypto(); self.rng=np.random.default_rng(seed)\n        self.available=set(available if available is not None else AUTHORITIES)\n        self.L=L; self.log=[]; self.replay_ev=set(); self.replay_cs=set(); self.executed=set()\n        self.wire_bytes={}  # Message sizes include ciphertext/tag and 96-bit GCM nonce; exclude implied AAD.\n        self.reg_ev=secrets.token_bytes(32); self.reg_cs=secrets.token_bytes(32)\n        self.pid_master=secrets.token_bytes(32); self.id_ev=\'ev_internal_reference_001\'\n        self.csid=\'CS_DEMO_01\'; self.op=\'CHARGE\'\n        self.sigkeys={a:self.pqc.sig_keygen() for a in AUTHORITIES}\n        self.resource=new_qss_resource(L,self.rng)\n        self.t0=int(time.time()); self.stage=\'NEW\'; self._last_mark=time.perf_counter()\n    def mark(self,phase, status=\'PASS\', **details):\n        now=time.perf_counter()\n        self.log.append({\'phase\':phase,\'status\':status,\'elapsed_ms\':round((now-self._last_mark)*1000,3),**details})\n        self._last_mark=now\n    def register(self):\n        require({\'A\',\'B\'}<=self.available,\'UC-A and UC-B required for online registration\')\n        rA=secrets.token_bytes(16)\n        ev_tag=mac(self.reg_ev,\'REG\',self.id_ev,rA)\n        # 32B tag -> 128 Bell-pair encodings, decoded with 2-qubit simulator\n        require(ct_eq(dense_transport_tag(ev_tag), mac(self.reg_ev,\'REG\',self.id_ev,rA)),\'EV Bell verification\')\n        self.pid=mac(self.pid_master,self.id_ev,secrets.token_bytes(16),self.t0+86400)[:16].hex()\n        rB=secrets.token_bytes(16)\n        cs_tag=mac(self.reg_cs,\'CSREG\',self.csid,rB)\n        require(ct_eq(dense_transport_tag(cs_tag),mac(self.reg_cs,\'CSREG\',self.csid,rB)),\'CS Bell verification\')\n        self.stage=\'REGISTERED\'; self.mark(\'registration\', bell_pairs_per_entity=128)\n    def authenticate(self, tamper_ev=False, tamper_cs=False):\n        require(self.stage==\'REGISTERED\',\'register first\')\n        require({\'A\',\'B\'}<=self.available,\'UC-A and UC-B required for online participant validation\')\n        self.ne=secrets.token_bytes(16); self.n_c=secrets.token_bytes(16)\n        self.ts_e=self.t0; self.ts_c=self.t0\n        self.ev_tag=mac(self.reg_ev,\'EV_AUTH\',self.pid,self.csid,self.op,self.ne,self.ts_e)\n        submitted=self.ev_tag\n        if tamper_ev: submitted=bytes([submitted[0]^1])+submitted[1:]\n        require(abs(int(time.time())-self.ts_e)<=300,\'stale EV timestamp\')\n        require((self.pid,self.ne) not in self.replay_ev,\'EV nonce replay\')\n        require(ct_eq(submitted,mac(self.reg_ev,\'EV_AUTH\',self.pid,self.csid,self.op,self.ne,self.ts_e)),\'EV proof rejected\')\n        self.replay_ev.add((self.pid,self.ne))\n        cs_tag=mac(self.reg_cs,\'CS_AUTH\',self.csid,self.pid,self.op,self.ne,self.n_c,self.ts_c)\n        if tamper_cs: cs_tag=bytes([cs_tag[0]^1])+cs_tag[1:]\n        require(abs(int(time.time())-self.ts_c)<=300,\'stale CS timestamp\')\n        require((self.csid,self.n_c) not in self.replay_cs,\'CS nonce replay\')\n        require(ct_eq(cs_tag,mac(self.reg_cs,\'CS_AUTH\',self.csid,self.pid,self.op,self.ne,self.n_c,self.ts_c)),\'CS proof rejected\')\n        self.replay_cs.add((self.csid,self.n_c))\n        ua_conf=mac(self.reg_ev,\'CS_CONF\',self.pid,self.csid,self.op,self.ne,self.n_c,self.ts_c,True)\n        require(ct_eq(ua_conf, mac(self.reg_ev,\'CS_CONF\',self.pid,self.csid,self.op,self.ne,self.n_c,self.ts_c,True)),\'EV cannot verify station attestation\')\n        self.stage=\'AUTHENTICATED\'; self.mark(\'cross_domain_authentication\')\n    def establish_session(self):\n        require(self.stage==\'AUTHENTICATED\',\'authenticate first\')\n        ek,dk=self.pqc.kem_keygen()\n        bind=encode(self.pid,self.csid,self.op,self.ne,self.n_c,self.ts_e,self.ts_c,ek)\n        proof=mac(self.reg_ev,\'KEM-EV\',hashlib.sha256(bind).digest())\n        require(ct_eq(proof,mac(self.reg_ev,\'KEM-EV\',hashlib.sha256(bind).digest())),\'KEM public key not authenticated\')\n        ct,ss_cs=self.pqc.encaps(ek)\n        T=encode(self.pid,self.csid,self.op,self.ne,self.n_c,self.ts_e,self.ts_c,ek,ct)\n        transcript_hash=hashlib.sha256(T).digest()\n        cs_proof=mac(self.reg_cs,\'KEM-CS\',transcript_hash)\n        require(ct_eq(cs_proof,mac(self.reg_cs,\'KEM-CS\',transcript_hash)),\'KEM ciphertext station authentication failed\')\n        ua_conf=mac(self.reg_ev,\'KEM-CONF\',transcript_hash,\'ACCEPT\')\n        require(ct_eq(ua_conf,mac(self.reg_ev,\'KEM-CONF\',transcript_hash,\'ACCEPT\')),\'KEM authority binding mismatch\')\n        ss_ev=self.pqc.decaps(dk,ct)\n        require(ct_eq(ss_ev,ss_cs),\'KEM shared secrets mismatch\')\n        self.sid=H(\'QSV2G-SID\',T)[:16].hex()\n        salt=hashlib.sha256(self.ne+self.n_c).digest()\n        prk_e=hkdf_extract(salt,ss_ev); prk_c=hkdf_extract(salt,ss_cs)\n        def key(prk,label): return hkdf_expand(prk, label.encode()+transcript_hash,32)\n        k_e2c=key(prk_e,\'QSV2G-E2C\'); k_c2e=key(prk_c,\'QSV2G-C2E\')\n        require(k_e2c != k_c2e,\'Directional traffic keys unexpectedly identical\')\n        k_ev_conf=key(prk_e,\'KC-EV\'); k_cs_conf=key(prk_c,\'KC-CS\')\n        mu_e=mac(k_ev_conf,\'EV-FINISHED\',self.sid,transcript_hash)\n        require(ct_eq(mu_e,mac(key(prk_c,\'KC-EV\'),\'EV-FINISHED\',self.sid,transcript_hash)),\'EV key confirmation failed\')\n        mu_c=mac(k_cs_conf,\'CS-FINISHED\',self.sid,transcript_hash,mu_e)\n        require(ct_eq(mu_c,mac(key(prk_e,\'KC-CS\'),\'CS-FINISHED\',self.sid,transcript_hash,mu_e)),\'CS key confirmation failed\')\n        self.e2c=Traffic(k_e2c); self.c2e=Traffic(k_c2e)\n        self.stage=\'SESSION\'; self.mark(\'session_establishment\', backend=self.pqc.kind, kem_ciphertext_bytes=len(ct), sid_bytes=len(bytes.fromhex(self.sid)))\n        # Ephemeral key material references are not kept as instance attributes.\n    def authorization_context(self, op=\'CHARGE\', limit_mwh=12000, lifetime_sec=600, tamper_ciphertext=False):\n        require(self.stage==\'SESSION\',\'session not established\')\n        require(op in (\'CHARGE\',\'DISCHARGE\') and limit_mwh>0,\'invalid operation or energy limit\')\n        self.op=op\n        self.limit={\'type\':\'ENERGY\',\'max\':int(limit_mwh),\'unit\':\'mWh\'}\n        self.vld=(self.t0-2,self.t0+int(lifetime_sec))\n        self.ctx=(self.pid,self.csid,self.sid,self.op,self.ne,self.n_c,self.ts_e,self.ts_c,self.limit,self.vld)\n        mctx=encode(*self.ctx)\n        ad=encode(\'QSV2G-CTX\',self.sid,self.csid)\n        pkt=self.c2e.encrypt(mctx,ad)\n        self.wire_bytes[\'context_offer\']=len(pkt[\'nonce\'])+len(pkt[\'ciphertext\'])\n        if tamper_ciphertext:\n            pkt[\'ciphertext\']=bytes([pkt[\'ciphertext\'][0]^1])+pkt[\'ciphertext\'][1:]\n        ev_received=self.c2e.decrypt(pkt,ad)\n        require(ct_eq(ev_received,mctx),\'EV context disagreement\')\n        require(self.vld[0]<=int(time.time())<=self.vld[1],\'context expired\')\n        self.ac=H(\'QSV2G-AC\',mctx)\n        self.consent=mac(self.reg_ev,\'AUTHZ-CONSENT\',self.ac)\n        ack={\'sid\':self.sid,\'ac\':self.ac,\'consent\':self.consent}\n        ack_pkt=self.e2c.encrypt(canon(ack),encode(\'QSV2G-CTX-ACK\',self.sid))\n        self.wire_bytes[\'context_ack\']=len(ack_pkt[\'nonce\'])+len(ack_pkt[\'ciphertext\'])\n        require(ct_eq(self.e2c.decrypt(ack_pkt,encode(\'QSV2G-CTX-ACK\',self.sid)),canon(ack)),\'EV context ack failed\')\n        self.stage=\'CONTEXT\'; self.mark(\'context_agreement\', context_hash=self.ac.hex())\n    def authority_verify(self, approvals=(\'A\',\'B\',\'C\'), revoked=False, bad_consent=False):\n        require(self.stage==\'CONTEXT\',\'context missing\')\n        require({\'A\',\'B\'}<=self.available,\'UC-A and UC-B online validation required (full system not arbitrary-one-outage tolerant)\')\n        require(self.vld[0]<=int(time.time())<=self.vld[1], \'authorization expired\')\n        consent=self.consent if not bad_consent else bytes([self.consent[0]^1])+self.consent[1:]\n        require(ct_eq(consent,mac(self.reg_ev,\'AUTHZ-CONSENT\',self.ac)),\'UC-A rejected EV consent\')\n        require(not revoked,\'EV revoked\')\n        require(ct_eq(H(\'QSV2G-AC\',encode(*self.ctx)),self.ac),\'context mismatch\')\n        self.decisions={a:(a in approvals and a in self.available) for a in AUTHORITIES}\n        self.accepted=[a for a in AUTHORITIES if self.decisions[a]]\n        require(len(self.accepted)>=2,\'fewer than two distinct approvals\')\n        self.stage=\'APPROVED\'; self.mark(\'authority_validation\', approved=\',\'.join(self.accepted))\n    def threshold_reconstruct(self,pair=(\'A\',\'B\'),tamper_share=False):\n        require(self.stage==\'APPROVED\',\'authority approval missing\')\n        require(len(pair)==2 and pair[0]!=pair[1] and set(pair)<=set(self.accepted),\'not an approved distinct authority pair\')\n        require(self.resource.state==\'UNUSED\',\'resource must be unused\')\n        self.pair=tuple(pair)\n        self.resource.reserve(self.ac,self.pair)\n        recovered=self.resource.reconstruct(self.ac,self.pair,self.rng,tamper=tamper_share)\n        self.reconstruction_hash=H(\'QSV2G-REC\',self.ac,self.resource.nu,recovered)\n        self.stage=\'RECONSTRUCTED\'; self.mark(\'qss_threshold\',pair=\'\'.join(pair),recovered=True, consumed=self.resource.state)\n    def issue_token(self,tamper_signature=False):\n        require(self.stage==\'RECONSTRUCTED\',\'QSS reconstruction missing\')\n        u,v=self.pair\n        statement=(\'QSV2G-AUTHZ\',self.ac,self.sid,self.resource.nu,u,v,self.reconstruction_hash,self.vld,\'ACCEPT\')\n        msg=encode(*statement)\n        signatures={a:self.pqc.sign(self.sigkeys[a][1],msg) for a in self.pair}\n        if tamper_signature:\n            signatures[u]=bytes([signatures[u][0]^1])+signatures[u][1:]\n        # UC-B verification; all public keys are trusted/bound to authority identities in simulation.\n        require(all(self.pqc.verify(self.sigkeys[a][0],msg,signatures[a]) for a in self.pair),\'invalid authority signature\')\n        self.token={\'statement\':statement,\'signatures\':signatures}\n        self.stage=\'TOKEN\'; self.mark(\'authorization_token\', signer_count=2, signature_bytes=sum(map(len,signatures.values())))\n    def protected_response(self,tamper_response=False, tamper_token_after_collector=False):\n        require(self.stage==\'TOKEN\',\'no accepted authorization proof\')\n        token={\'statement\':self.token[\'statement\'],\'signatures\':dict(self.token[\'signatures\'])}\n        if tamper_token_after_collector:\n            signer=self.pair[0]\n            sig=token[\'signatures\'][signer]\n            token[\'signatures\'][signer]=bytes([sig[0]^1])+sig[1:]\n        resp={\'sid\':self.sid,\'pid\':self.pid,\'csid\':self.csid,\'ac\':self.ac,\n              \'op\':self.op,\'ne\':self.ne,\'nc\':self.n_c,\'lim\':self.limit,\n              \'vld\':self.vld,\'decision\':True,\'auth_token\':token}\n        ad=encode(\'QSV2G-RESP\',self.sid,self.csid)\n        pkt=self.c2e.encrypt(canon(resp),ad)\n        self.wire_bytes[\'authorization_response\']=len(pkt[\'nonce\'])+len(pkt[\'ciphertext\'])\n        if tamper_response:\n            pkt[\'ciphertext\']=pkt[\'ciphertext\'][:-1]+bytes([pkt[\'ciphertext\'][-1]^1])\n        received=parse_canon(self.c2e.decrypt(pkt,ad))\n        # EV validates independently from received wire bytes, never from a local CS plaintext.\n        require(received[\'sid\']==self.sid and received[\'pid\']==self.pid and received[\'csid\']==self.csid,\'identity/session mismatch\')\n        require(received[\'op\']==self.op and received[\'ne\']==self.ne and received[\'nc\']==self.n_c,\'nonce/op mismatch\')\n        require(received[\'lim\']==self.limit and received[\'vld\']==self.vld and ct_eq(received[\'ac\'],self.ac),\'transaction context mismatch\')\n        require(self.vld[0]<=int(time.time())<=self.vld[1],\'authorization expired at EV\')\n        require(received[\'decision\'] is True,\'EV received a reject response\')\n        wire_token=received[\'auth_token\']\n        statement=wire_token[\'statement\']\n        require(statement[1]==self.ac and statement[2]==self.sid and statement[3]==self.resource.nu,\'signed statement context mismatch\')\n        # The EV cannot calculate or know h_rec; it can only verify the signed attestation.\n        require(isinstance(statement[6],bytes) and len(statement[6])==32 and statement[7]==self.vld and statement[8]==\'ACCEPT\',\'signed statement inconsistency\')\n        wire_pair=(statement[4],statement[5])\n        require(len(set(wire_pair))==2 and all(a in AUTHORITIES for a in wire_pair),\'invalid token signers\')\n        msg=encode(*statement)\n        require(all(self.pqc.verify(self.sigkeys[a][0],msg,wire_token[\'signatures\'][a]) for a in wire_pair),\'EV token signature invalid\')\n        require((self.sid,self.ac,self.resource.nu) not in self.executed,\'token already executed\')\n        self.stage=\'EV_ACCEPTED\'; self.mark(\'ev_token_verification\', decision=\'ACCEPT\')\n    def execute(self,power_w=1200,seconds=30,tick_s=5):\n        require(self.stage==\'EV_ACCEPTED\',\'EV did not accept token\')\n        tid=(self.sid,self.ac,self.resource.nu)\n        require(tid not in self.executed,\'repeat authorization execution\')\n        self.executed.add(tid)  # atomic authorization consumption modeled by memory store\n        nx=secrets.token_bytes(16)\n        req=encode(\'QSV2G-START\',self.sid,self.ac,self.resource.nu,self.op,nx)\n        pkt=self.e2c.encrypt(req,encode(\'QSV2G-START\',self.sid))\n        self.wire_bytes[\'execution_start\']=len(pkt[\'nonce\'])+len(pkt[\'ciphertext\'])\n        require(ct_eq(self.e2c.decrypt(pkt,encode(\'QSV2G-START\',self.sid)),req),\'start message rejected\')\n        require(self.vld[0]<=int(time.time())<=self.vld[1],\'validity expired before execution\')\n        # Integer micro-Wh units avoid floating-point serialization or rounding ambiguity.\n        energy_uwh=0; limit_uwh=self.limit[\'max\']*1000; steps=0\n        logical_time=self.t0\n        for _ in range(math.ceil(seconds/tick_s)):\n            dt=min(tick_s,seconds-steps*tick_s)\n            if dt<=0 or logical_time > self.vld[1]: break\n            increment=round(power_w*dt/3600*1_000_000)\n            energy_uwh+=min(increment,limit_uwh-energy_uwh)\n            steps+=1; logical_time+=dt\n            if energy_uwh>=limit_uwh: break\n        require(energy_uwh<=limit_uwh,\'energy ceiling exceeded\')\n        self.energy_mwh=energy_uwh/1000\n        self.stage=\'EXECUTED\'; self.mark(\'energy_execution\', energy_mwh=self.energy_mwh, ticks=steps, op=self.op)\n    def terminate(self, interrupted=False):\n        require(self.stage==\'EXECUTED\',\'no active or completed execution\')\n        status=\'INTERRUPTED\' if interrupted else \'COMPLETED\'\n        record={\'sid\':self.sid,\'ac\':self.ac,\'nu\':self.resource.nu,\'op\':self.op,\n                \'status\':status,\'ended_at\':int(time.time()), \'metered_uWh\':int(round(self.energy_mwh*1000))}\n        ad=encode(\'QSV2G-TERM\',self.sid,self.csid)\n        pkt=self.c2e.encrypt(canon(record),ad)\n        self.wire_bytes[\'termination\']=len(pkt[\'nonce\'])+len(pkt[\'ciphertext\'])\n        require(ct_eq(self.c2e.decrypt(pkt,ad),canon(record)),\'termination record rejected\')\n        self.stage=\'TERMINATED\'; self.mark(\'termination\',status=status)\n        return record\n\ndef run_demo(crypto=None, pair=(\'A\',\'B\'), seed=2026,L=162,**fault):\n    p=QSV2G(crypto=crypto,seed=seed,L=L,available=fault.get(\'available\'))\n    try:\n        p.register()\n        p.authenticate(tamper_ev=fault.get(\'tamper_ev\',False),tamper_cs=fault.get(\'tamper_cs\',False))\n        p.establish_session()\n        p.authorization_context(op=fault.get(\'op\',\'CHARGE\'),limit_mwh=fault.get(\'limit_mwh\',12000),\n                                lifetime_sec=fault.get(\'lifetime_sec\',600),tamper_ciphertext=fault.get(\'tamper_context\',False))\n        p.authority_verify(approvals=fault.get(\'approvals\',AUTHORITIES),revoked=fault.get(\'revoked\',False),bad_consent=fault.get(\'bad_consent\',False))\n        p.threshold_reconstruct(pair=pair,tamper_share=fault.get(\'tamper_share\',False))\n        p.issue_token(tamper_signature=fault.get(\'tamper_signature\',False))\n        p.protected_response(tamper_response=fault.get(\'tamper_response\',False),tamper_token_after_collector=fault.get(\'tamper_token_after_collector\',False))\n        p.execute()\n        p.terminate()\n        return p\n    except ProtocolAbort as e:\n        p.mark(p.stage,\'REJECTED\',reason=str(e))\n        return p\n'
assert hashlib.sha256(REFERENCE_CORE.encode()).hexdigest()==REFERENCE_SHA256, "Reference code changed"
exec(compile(REFERENCE_CORE,"qsv2g_reference_core.py","exec"),globals())
pqc=PQCrypto()
print("Core SHA256",REFERENCE_SHA256,"PQC provider:",pqc.kind)

## 2. Evidence harness
Every case has: a stable ID, target requirement, expected vs observed outcome, attack boundary, detailed observation and a Boolean *match to expectation*. Do not interpret `match_to_expectation=True` as a demonstrated security guarantee.

In [ ]:
CASES=[]
def observe(fn):
    try:
        ans=fn()
        return ('ACCEPTED',str(ans)[:500] if ans is not None else '')
    except ProtocolAbort as e:
        return ('REJECTED',str(e)[:500])
    except Exception as e:
        return ('ERROR',type(e).__name__+': '+str(e)[:500])

def record(cid, group, spr, model, expected, observed, notes='', detail=''):
    assert not any(x['case_id']==cid for x in CASES),'Repeated case ID '+cid
    row={'case_id':cid,'group':group,'requirement':spr,'model_boundary':model,
        'expected':expected,'observed':observed,'matches_expectation':expected==observed,
        'interpretation':notes,'observed_detail':detail}
    CASES.append(row)
    print(f"{cid:>5} {('MATCH' if expected==observed else 'MISMATCH'):>8} observed={observed:<12} expected={expected:<12} {notes[:82]}")
    return row

def attack(cid,grp,spr,fn,expected='REJECTED',notes='',model='simulated_protocol'):
    observed,detail=observe(fn)
    return record(cid,grp,spr,model,expected,observed,notes,detail)

def setup_to(phase='CONTEXT',L=9,op='CHARGE',available=None):
    p=QSV2G(crypto=pqc,L=L,available=available)
    p.register()
    if phase=='REGISTERED': return p
    p.authenticate()
    if phase=='AUTHENTICATED':return p
    p.establish_session()
    if phase=='SESSION':return p
    p.authorization_context(op=op)
    if phase=='CONTEXT':return p
    p.authority_verify()
    if phase=='APPROVED':return p
    p.threshold_reconstruct(pair=('A','B'))
    if phase=='RECONSTRUCTED':return p
    p.issue_token()
    if phase=='TOKEN':return p
    p.protected_response()
    return p

def result_stage(fn):
    p=fn()
    return ('ACCEPTED' if p.stage=='TERMINATED' else 'REJECTED',p.log[-1].get('reason','') if p.log else '')

# Keep reference-case names and case IDs stable for manuscript evidence lookup.

## A. Participant authentication, policy, consent, and availability
These are direct calls to the reference prototype, including intentionally malformed inputs. A successful invalid-input rejection is limited evidence, not security against every active or quantum adversary.

In [ ]:
for cid,title,kwargs,spr in [
 ('A01','EV credential/proof tamper',{'tamper_ev':True},'SPR1'),
 ('A02','CS credential/proof tamper',{'tamper_cs':True},'SPR1'),
 ('A03','Encrypted context tamper',{'tamper_context':True},'SPR5,SPR8'),
 ('A04','Revoked participant',{'revoked':True},'SPR11'),
 ('A05','Invalid EV consent',{'bad_consent':True},'SPR2'),
 ('A06','One approving authority',{'approvals':('A',)},'SPR2,SPR3'),
 ('A07','No approving authorities',{'approvals':()},'SPR2,SPR3'),
 ('A08','QSS tampered during reconstruction',{'tamper_share':True},'SPR4'),
 ('A09','Signature tamper before collector',{'tamper_signature':True},'SPR9'),
 ('A10','Protected response ciphertext tamper',{'tamper_response':True},'SPR8'),
 ('A11','Token signature tamper after collector',{'tamper_token_after_collector':True},'SPR9'),
 ('A12','Expired transaction',{'lifetime_sec':-1},'SPR5,SPR13'),
 ('A13','Invalid zero energy limit',{'limit_mwh':0},'SPR6'),
 ('A14','Unavailable UC-A',{'available':('B','C'),'pair':('B','C')},'SPR10'),
 ('A15','Unavailable UC-B',{'available':('A','C'),'pair':('A','C')},'SPR10'),
]:
    kw=dict(kwargs);pair=kw.pop('pair',('A','B'))
    flow=run_demo(crypto=pqc,L=7,pair=pair,**kw)
    observed='ACCEPTED' if flow.stage=='TERMINATED' else 'REJECTED'
    record(cid,'A',spr,'reference_run_demo','REJECTED',observed,title,flow.log[-1].get('reason','') if flow.log else '')

p=run_demo(crypto=pqc,L=7,pair=('A','B'),available=('A','B'))
record('A16','A','SPR10','reference_run_demo','ACCEPTED','ACCEPTED' if p.stage=='TERMINATED' else 'REJECTED',
       'UC-C unavailable; AB can complete when A and B are online')

for idx,(op,pair) in enumerate(itertools.product(('CHARGE','DISCHARGE'),itertools.combinations(AUTHORITIES,2)),1):
    flow=run_demo(crypto=pqc,L=7,op=op,pair=pair)
    record(f'A{16+idx:02d}','A','SPR1,SPR2,SPR3,SPR6','reference_run_demo','ACCEPTED',
           'ACCEPTED' if flow.stage=='TERMINATED' else 'REJECTED',
           f'Happy-path {op} with pair {"".join(pair)}',flow.log[-1].get('reason','') if flow.log else '')

p=setup_to('REGISTERED',L=7);p.t0-=3600
attack('A23','A','SPR13',p.authenticate,notes='Old EV timestamp rejected by time window')

# Force the same EV or CS nonce only inside this *unit-test perturbation*; no real network event.
for cid,who in [('A24','EV'),('A25','CS')]:
    p=setup_to('REGISTERED',L=7)
    fixed=b'Q'*16
    if who=='EV':p.replay_ev.add((p.pid,fixed))
    else:p.replay_cs.add((p.csid,fixed))
    with patch.object(secrets,'token_bytes',return_value=fixed):
        attack(cid,'A','SPR5,SPR13',p.authenticate,notes=f'Forced duplicate {who} nonce rejected',model='injected_randomness_unit_test')

p=QSV2G(crypto=pqc,L=7)
attack('A26','A','SPR1',p.authenticate,notes='Authentication before registration is rejected')
print('A suite complete:',len(CASES),'cases')

## B. Transaction binding, AEAD, and freshness
Mutating the stored authorization tuple is an explicit fault-injection test against the context digest. Separate tests exercise actual authenticated encryption with deliberately modified packet or associated data.

In [ ]:
# All transaction fields are bound by the canonical hash. The attack is a modeled change *inside* the pending CS context object.
fields=[
 ('PID',0,lambda v:v+'-other'),('CSID',1,lambda v:v+'-other'),('SID',2,lambda v:v+'-other'),
 ('OP',3,lambda v:'DISCHARGE' if v=='CHARGE' else 'CHARGE'),
 ('NE',4,lambda v:bytes([v[0]^1])+v[1:]),('NC',5,lambda v:bytes([v[0]^1])+v[1:]),
 ('TS_E',6,lambda v:v+1),('TS_CS',7,lambda v:v+1),
 ('LIM',8,lambda v:dict(v,max=v['max']+1)),
 ('VLD',9,lambda v:(v[0],v[1]+1)),
]
for k,(field,ind,mut) in enumerate(fields,1):
    p=setup_to('CONTEXT',L=6);tmp=list(p.ctx);tmp[ind]=mut(tmp[ind]);p.ctx=tuple(tmp)
    attack(f'B{k:02d}','B','SPR2,SPR5,SPR6,SPR13',p.authority_verify,notes=f'Changed bound field {field}',model='internal_context_mutation')

p=setup_to('CONTEXT',L=6);p.ac=bytes([p.ac[0]^1])+p.ac[1:]
attack('B11','B','SPR2,SPR5',p.authority_verify,notes='Altered authorization digest violates authenticated consent')
record('B12','B','SPR5','canonical_length_prefix','DISTINCT',
       'DISTINCT' if encode(b'a',b'bc')!=encode(b'ab',b'c') else 'COLLISION',
       'Different field boundaries have different canonical byte encodings')

key=secrets.token_bytes(32);t=Traffic(key);ad=encode('QSV2G-TEST','sid-1');pkt=t.encrypt(b'payload',ad)
attack('B13','B','SPR8',lambda:t.decrypt({'nonce':pkt['nonce'],'ciphertext':pkt['ciphertext'][:-1]+bytes([pkt['ciphertext'][-1]^1])},ad),notes='Authenticated ciphertext modification rejected',model='actual_AESGCM')
record('B14','B','SPR8','actual_AESGCM','ACCEPTED',observe(lambda:t.decrypt(pkt,ad))[0],'Untampered packet decrypts')
attack('B15','B','SPR5,SPR13',lambda:t.decrypt(pkt,ad),notes='Duplicate accepted packet nonce rejected',model='actual_AESGCM')
pkt2=t.encrypt(b'payload2',ad)
attack('B16','B','SPR8',lambda:t.decrypt(pkt2,encode('QSV2G-TEST','other-session')),notes='Wrong associated data fails authentication',model='actual_AESGCM')
pkt3=t.encrypt(b'payload3',ad)
attack('B17','B','SPR8',lambda:t.decrypt({'nonce':b'bad','ciphertext':pkt3['ciphertext']},ad),notes='Malformed nonce length rejected',model='actual_AESGCM')

p=setup_to('SESSION',L=6)
record('B18','B','SPR8','directional_KDF','DISTINCT','DISTINCT' if p.e2c.key!=p.c2e.key else 'SAME',
       'EV-to-CS and CS-to-EV derived keys differ')

# Expose the nonce state restart hazard without silently treating it as a PASS of confidentiality.
k=secrets.token_bytes(32);pa=Traffic(k).encrypt(b'one',ad);pb=Traffic(k).encrypt(b'two',ad)
record('B19','B','SPR5,SPR8','deployment_limitation','EXPOSED','EXPOSED' if pa['nonce']==pb['nonce'] else 'NOT_DETECTED',
       'KEY/COUNTER RESTART HAZARD: repeated AES-GCM nonce under same traffic key')

# This test changes station-held mutable state after QSS/token issuance, a stronger fault than external message tampering.
p=setup_to('TOKEN',L=6,op='CHARGE');signed_op=p.ctx[3];p.op='DISCHARGE';status,reason=observe(p.protected_response)
record('B20','B','SPR2,SPR5,SPR6','trusted_object_state_corruption','EXPOSED',
       'EXPOSED' if status=='ACCEPTED' and p.op!=signed_op else 'REJECTED',
       'Check whether EV-side reference simulator independently keeps original OP after station state mutation; NOT a network attacker test.',reason)
print('B suite complete:',len(CASES),'cases cumulative')

## C. Quantum resource integrity, contribution consistency, dealer behavior
This suite exercises real `QSSResource` objects and exact 27-dimensional statevectors. It explicitly demonstrates the existing **malicious-dealer** limitation and the difference between stabilizer consistency and verification of the intended logical value.

In [ ]:
rng=np.random.default_rng(20261026)
for pair in itertools.combinations(AUTHORITIES,2):
    results=[]
    for sym in range(3):
        for _ in range(30):
            xu,xv=measure_pair(qss_state(sym),pair,rng)
            results.append(recover_symbol(pair,xu,xv)==sym)
    record('C'+''.join(pair),'C','SPR3,SPR4','ideal_statevector','ACCEPTED',
           'ACCEPTED' if all(results) else 'INCONSISTENT',
           f'Ideal pair {pair}: {sum(results)}/{len(results)} measured trits correct')

marginal=[]
for s in range(3):
    psi=qss_state(s)
    for i,a in enumerate(AUTHORITIES):
        delta=float(np.linalg.norm(reduced_density(psi,i)-np.eye(3)/3))
        marginal.append({'s':s,'authority':a,'distance_from_I3_over_3':delta})
record('C04','C','SPR7','ideal_one_share_marginal','IDEAL_PRIVACY','IDEAL_PRIVACY' if all(x['distance_from_I3_over_3']<1e-12 for x in marginal) else 'NONUNIFORM',
       'One ideal quantum share has marginal I3/3; public commitments and protocol metadata are excluded')

for pair in itertools.combinations(AUTHORITIES,2):
    for dealer in AUTHORITIES:
        q=new_qss_resource(L=7);q.reserve(b'AC',pair)
        s=q.dealer_secrets[dealer][0];q.states[dealer][0]=qss_state((s+1)%3)
        status,why=observe(lambda:q.reconstruct(b'AC',pair,rng))
        cid='C'+str(5+list(itertools.combinations(AUTHORITIES,2)).index(pair)*3+AUTHORITIES.index(dealer)).zfill(2)
        record(cid,'C','SPR4','post_commitment_state_substitution','REJECTED',status,
               f'Altered {dealer} quantum state for pair {"".join(pair)}',why+'; resource_state='+q.state)

q=new_qss_resource(L=7);q.commitments['A']=bytes(32);q.reserve(b'AC',('A','B'))
attack('C14','C','SPR4',lambda:q.reconstruct(b'AC',('A','B'),rng),notes='Tampered prior commitment rejected')
record('C15','C','SPR4','resource_lifecycle','CONSUMED',q.state,'Failed reconstruction consumes measured states')
q=new_qss_resource(L=7);q.reserve(b'AC',('A','B'))
attack('C16','C','SPR3',lambda:q.reserve(b'AC2',('A','C')),notes='Double reservation rejected')
attack('C17','C','SPR4',lambda:q.reconstruct(b'OTHER',('A','B'),rng),notes='Wrong AC cannot reconstruct reserved quantum resource')
q2=new_qss_resource(L=7)
attack('C18','C','SPR3',lambda:q2.reserve(b'AC',('A',)),notes='Single share cannot reserve an authorized reconstruction pair')
q2.reserve(b'AC',('A','B'));q2.reconstruct(b'AC',('A','B'),rng)
attack('C19','C','SPR4',lambda:q2.reserve(b'AC2',('A','C')),notes='Consumed resource is unavailable for reuse')

q=new_qss_resource(L=7)
q.dealer_secrets['C']=[(s+1)%3 for s in q.dealer_secrets['C']]
q.states['C']=[qss_state(s) for s in q.dealer_secrets['C']]
q.commitments['C']=H('QSV2G-QSS-COM','C',q.nu,q.dealer_secrets['C'])
q.reserve(b'AC',('A','B'))
status,why=observe(lambda:q.reconstruct(b'AC',('A','B'),rng))
record('C20','C','SPR4','known_limitation_dishonest_dealer','EXPOSED','EXPOSED' if status=='ACCEPTED' else 'NOT_DETECTED',
       'Self-consistent dealer substitutes secret and commitment without detection',why)

st=qss_consistency()
residuals=max(x['x_stabilizer_error'] for x in st)+max(x['z_stabilizer_error'] for x in st)
record('C21','C','SPR4','known_limitation_stabilizers','EXPOSED','EXPOSED' if residuals<1e-12 else 'NOT_DETECTED',
       'All three encoded symbols satisfy same tested stabilizers; logical intent not identified')

# Low-entropy public commitments can reveal short dealer secrets through offline enumeration.
q=new_qss_resource(L=1);stored=q.commitments['B']
candidates=[[s] for s in range(3) if ct_eq(H('QSV2G-QSS-COM','B',q.nu,[s]),stored)]
record('C22','C','SPR7','parameterization_caveat','EXPOSED','EXPOSED' if len(candidates)==1 else 'NOT_DETECTED',
       'L=1 low-entropy commitment dictionary recovery; NOT an attack on the full L=162 setting')

# Distinguish lack of a corrupt-dealer proof from benign testing.
p=setup_to('APPROVED',L=6);p.resource.reserve(p.ac,('A','B'))
attack('C23','C','SPR3',p.issue_token,notes='Honest simulator cannot issue token before reconstruction')
print('C suite complete:',len(CASES),'cases cumulative')

## D. Authority signatures, replayed authorization, execution and trust boundaries
These tests include actual ML-DSA verification, a one-signer forgery attempt and the previously demonstrated two-colluding-signer attestation limitation. They do not constitute real-world authority compromise simulations.

In [ ]:
# Invalid second authority: the verification key of B must reject A's signature.
p=setup_to('APPROVED',L=6);fake=('QSV2G-AUTHZ',p.ac,p.sid,p.resource.nu,'A','B',secrets.token_bytes(32),p.vld,'ACCEPT')
msg=encode(*fake);sig_a=pqc.sign(p.sigkeys['A'][1],msg)
p.token={'statement':fake,'signatures':{'A':sig_a,'B':sig_a}};p.stage='TOKEN'
attack('D01','D','SPR9',p.protected_response,notes='Authority A signature cannot impersonate authority B')

# Distinct valid signatures are accepted even if the pair illicitly skips quantum reconstruction.
p=setup_to('APPROVED',L=6);fake=('QSV2G-AUTHZ',p.ac,p.sid,p.resource.nu,'A','B',secrets.token_bytes(32),p.vld,'ACCEPT')
msg=encode(*fake)
p.token={'statement':fake,'signatures':{'A':pqc.sign(p.sigkeys['A'][1],msg),'B':pqc.sign(p.sigkeys['B'][1],msg)}}
p.stage='TOKEN';status,why=observe(p.protected_response)
record('D02','D','SPR3,SPR4','known_limitation_two_corrupt_signers','EXPOSED',
       'EXPOSED' if status=='ACCEPTED' and p.resource.state=='UNUSED' else 'NOT_DETECTED',
       'Two actual signing keys attest to reconstruction they deliberately skipped; exceeds <=1-corrupt-authority threat bound.',why)

p=setup_to('TOKEN',L=6)
t=list(p.token['statement']);t[6]=bytes([t[6][0]^1])+t[6][1:];p.token['statement']=tuple(t)
attack('D03','D','SPR2,SPR9',p.protected_response,notes='Modified signed reconstruction digest rejected')

p=setup_to('TOKEN',L=6)
p.sid+='-other'
attack('D04','D','SPR2,SPR5',p.protected_response,notes='Signed statement for another SID rejected')

p=setup_to('TOKEN',L=6)
p.resource.nu+='-other'
attack('D05','D','SPR5',p.protected_response,notes='Signed statement for another quantum-resource ID rejected')

p=setup_to('EV_ACCEPTED',L=6);p.execute();p.terminate()
attack('D06','D','SPR5,SPR6',p.execute,notes='Repeated energy execution after termination rejected')

p=setup_to('EV_ACCEPTED',L=6);p.execute(power_w=999999,seconds=90,tick_s=5)
record('D07','D','SPR6','integer_energy_counter','WITHIN_LIMIT',
       'WITHIN_LIMIT' if p.energy_mwh <= p.limit['max'] and p.energy_mwh>=0 else 'OVER_LIMIT',
       'Integer micro-Wh counter caps modeled transfer; not a physical meter test',
       f'energy_mwh={p.energy_mwh}; limit_mwh={p.limit["max"]}')

p=setup_to('EV_ACCEPTED',L=6);p.execute();rec=p.terminate(interrupted=True)
record('D08','D','SPR6','simulated_termination','INTERRUPTED',rec['status'],
       'Interrupted termination produces authenticated simulated record')

p=setup_to('TOKEN',L=6);p.vld=(p.t0-10,p.t0-5)
attack('D09','D','SPR5,SPR13',p.protected_response,notes='EV rejects expired authorization after signing')

p=setup_to('APPROVED',L=6)
attack('D10','D','SPR3',p.issue_token,notes='Honest issuance fails before quantum reconstruction')

# Observe a cryptographically good token with altered station-held OP but unchanged signed AC.
p=setup_to('TOKEN',L=6);p.op='DISCHARGE'
status,why=observe(p.protected_response)
record('D11','D','SPR2,SPR6','shared_state_simulation_gap','EXPOSED',
       'EXPOSED' if status=='ACCEPTED' else 'REJECTED',
       'Reference model lacks an independent EV copy of original accepted OP under internal station-state mutation; NOT proof of network exploit.',why)
print('D suite complete:',len(CASES),'cases cumulative')

## 3. Security/privacy requirements mapping (SPR1–SPR13)
A requirement is `PARTIALLY_TESTED`, `LIMITATION_DEMONSTRATED`, or `NOT_ESTABLISHED`. These categories are **scientific scope statements**, not numerical security scores. Citations to cases are traceability links; all tests occur within the supplied software model.

In [ ]:
requirements=[
 ('SPR1','Participant mutual authentication','PARTIALLY_TESTED','A01,A02,A17,A18,A19,A20,A21,A22','HMAC-based authority-mediated credential validation is simulated; physical and quantum network threats excluded.'),
 ('SPR2','Context-bound authorization','PARTIALLY_TESTED','A05,A06,B01-B12,D03,D04,D11','Canonical AC checked within shared simulation; D11 shows absence of independently held EV intent under corrupted station state.'),
 ('SPR3','Threshold enforcement','LIMITATION_DEMONSTRATED','A06,A07,C16,C18,C23,D02,D10','Honest gate requires reconstruction; two colluding valid signers can bypass it while EV accepts signed attestation.'),
 ('SPR4','QSS contribution consistency','LIMITATION_DEMONSTRATED','A08,C05-C15,C20,C21,D02','Commitment mismatch caught; self-consistent dishonest dealer and logical-state ambiguity persist.'),
 ('SPR5','Replay and context substitution','PARTIALLY_TESTED','A03,A12,A24,A25,B01-B19,D04,D05,D06,D09','Replay and mismatches tested; safe crash-persistent key/state recovery is not implemented.'),
 ('SPR6','Fine-grained operation control','PARTIALLY_TESTED','A13,A17-A22,B04,B09,D07,D08,D11','CHARGE/DISCHARGE and simulated integer energy cap; no physical trusted meter or controller.'),
 ('SPR7','EV identity privacy','NOT_ESTABLISHED','C04,C22','Ideal quantum marginal is I/3; pseudonym unlinkability, metadata privacy and public-commitment secrecy not established.'),
 ('SPR8','Protected CS-to-EV response','PARTIALLY_TESTED','A03,A10,B13-B19','AES-GCM verifies tested messages; persistent nonce safety and real network interface not established.'),
 ('SPR9','Single-signer forgery resistance','PARTIALLY_TESTED','A09,A11,D01,D03','Invalid second signature rejected under ML-DSA assumptions; no formal quantum-adversary proof.'),
 ('SPR10','Conditional pair availability','LIMITATION_DEMONSTRATED','A14,A15,A16','UC-C outage tolerated in one configuration; complete online service still depends on UC-A and UC-B.'),
 ('SPR11','Revocation enforcement','PARTIALLY_TESTED','A04','Boolean revoked flag rejected; no deployed revocation data feed or policy-distribution consistency.'),
 ('SPR12','Controlled accountability','NOT_ESTABLISHED','','UC-A stores mapping in abstract model; lawful disclosure controls, audit and access enforcement not implemented.'),
 ('SPR13','Fresh session binding','PARTIALLY_TESTED','A12,A23,A24,A25,B01-B11,D09','Timestamps/nonces/AC bound in simulator; full state persistence and clock-skew operations not established.'),
]
reqdf=pd.DataFrame(requirements,columns=['requirement','name','assessment','case_ids','scientific_qualification'])
reqdf.to_csv(OUT/'security_requirements_matrix.csv',index=False)
results=pd.DataFrame(CASES)
results.to_csv(OUT/'security_attack_results.csv',index=False)
known=results[results.model_boundary.str.contains('limitation|hazard|caveat|state_corruption|simulation_gap',case=False,regex=True,na=False)].copy()
known.to_csv(OUT/'security_limitations.csv',index=False)
print(reqdf[['requirement','name','assessment']].to_string(index=False))
print('Case count:',len(results),' mismatches:',int((~results.matches_expectation).sum()))

## 4. Experiment trace, provenance, ZIP export, and strict interpretation
The ZIP includes **all** cases, including any mismatches and any observed vulnerabilities. Source code is hashed and is embedded in this notebook; no network traffic is sent to GitHub. A JSON summary records the actual backend, environment, and qualitative limitations.

In [ ]:
import zipfile, csv

def package_ver(pkg):
    try:return metadata.version(pkg)
    except metadata.PackageNotFoundError:return None

env={'utc_timestamp':pd.Timestamp.now(tz='UTC').isoformat(),
     'python':sys.version,'platform':platform.platform(),'run_location_hint':'Kaggle working directory' if ON_KAGGLE else 'local Python',
     'openssl':subprocess.run(['openssl','version'],capture_output=True,text=True).stdout.strip(),
     'pqc_backend':pqc.kind,'numpy':package_ver('numpy'),'pandas':package_ver('pandas'),
     'cryptography':package_ver('cryptography'),'pqcrypto':package_ver('pqcrypto'),
     'source_code_sha256':REFERENCE_SHA256,
     'section_iv_reference_sha256':'4ca43b1f6debd2e2bf314ff9adf74f0ad08ba1702e1055eca1943b9533290ef2',
     'randomness':'Cryptographic secrets from secrets module; deterministic NumPy seed for sample tests',
     'kaggle_version_or_run_id':'NOT_CAPTURED_AUTOMATICALLY — attach Kaggle run URL/revision separately'}
(OUT/'environment.json').write_text(json.dumps(env,indent=2),encoding='utf-8')

# All participants in stage traces are synthetic and protocol secrets are NOT exported.
summary={'project':'QSV2G','stage':'02_security_analysis','case_count':len(results),
         'expectations_matched':int(results.matches_expectation.sum()),
         'mismatches':int((~results.matches_expectation).sum()),
         'assessments':reqdf.assessment.value_counts().to_dict(),
         'interpretation':'Matching an expected vulnerability is NOT a security success. Software tests are not a proof.',
         'reference_core_sha256':REFERENCE_SHA256,'reported_backend':pqc.kind,
         'status':'PASS_WITH_DOCUMENTED_LIMITATIONS' if results.matches_expectation.all() else 'REVIEW_REQUIRED',
         'known_boundaries':['No malicious-secure VQSS','No independent EV proof of QSS reconstruction',
              'Home and visited utility availability dependencies', 'Crash-persistent AEAD nonce/state not implemented',
              'No physical quantum or V2G hardware', 'No formal quantum-adversary proof']}
(OUT/'security_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')

report=['# QSV2G Stage 02 — Security analysis evidence report','',
        f"**Status:** {summary['status']}",
        f"**Cases:** {summary['case_count']}, matched expectations: {summary['expectations_matched']}, mismatches: {summary['mismatches']}",
        '','These are fault-injection and functional evidence under the stated simulator trust boundaries, NOT a cryptographic proof.',
        '','## Requirement assessments','','```csv\n'+reqdf.to_csv(index=False)+'\n```',
        '','## Case expectation mismatches (investigate; not suppressed)','',
        '```csv\n'+results[~results.matches_expectation].to_csv(index=False)+'\n```' if not results.matches_expectation.all() else 'None observed.',
        '','## Documented limitations','',
        '- Low-entropy commitments are exhaustively enumerable for deliberately short L=1; not an attack on the default 162-trit setting.',
        '- Token-signature verification does not independently prove quantum reconstruction.',
        '- Software shares mutable EV/CS protocol state; the internal corruption tests are **not remote exploit demonstrations**.',
        '- Two colluding authority signers exceed a model with at most one malicious authority.',
        '- UC-A/UC-B service dependencies and nonce-state restart hazards persist.',
        '- No malicous-secure VQSS, unconditional anonymity or fault-tolerant physical network is established.']
(OUT/'security_report.md').write_text(chr(10).join(report),encoding='utf-8')

# Manifest of exported evidence, excluding the manifest itself.
artifacts=[]
for f in sorted(OUT.iterdir()):
    if f.is_file():artifacts.append({'filename':f.name,'size_bytes':f.stat().st_size,'sha256':hashlib.sha256(f.read_bytes()).hexdigest()})
(OUT/'artifact_manifest.json').write_text(json.dumps(artifacts,indent=2),encoding='utf-8')
zip_path=WORK/'qsv2g_stage02_security_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,arcname=f.name)
with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None
print('STAGE RESULT:',summary['status'])
print('Cases:',summary['case_count'],'matched expected:',summary['expectations_matched'],'mismatches:',summary['mismatches'])
print('Evidence ZIP:',zip_path)
if summary['mismatches']:
    print('CAUTION: Review the unexpected cases in security_attack_results.csv; do not claim Stage 02 passed.')

## What to send back

After the notebook prints `STAGE RESULT: PASS` or `STAGE RESULT: REVIEW_REQUIRED`, download **`qsv2g_stage02_security_results.zip`** from Kaggle Outputs and upload it here. Also save the **executed notebook** or its immutable Kaggle run URL/revision if available. `REVIEW_REQUIRED` does **not** mean the ZIP should be discarded; unexpected results can be scientifically important.

**If the notebook cannot install `pqcrypto`:** Confirm **Internet ON** in Kaggle notebook settings, restart the notebook session, and select **Run All**. We require real ML-KEM-768/ML-DSA-65; please do not substitute mock cryptography.

The experiment is limited to software-model attack injection. We will use its results to populate Section V, not claim formal or physical security proofs.